In [4]:
import os

import mlflow
from utils.clnt_utils import get_databricks_ai_gateway_client, get_openai_client, get_ai_gateway_model_names, is_databricks_ai_gateway_client
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Configure MLflow
mlflow.set_tracking_uri("http://localhost:5000")

use_ai_gateway = is_databricks_ai_gateway_client()

# Verify which client to use
if use_ai_gateway:
    client = get_databricks_ai_gateway_client()
    model_name = get_ai_gateway_model_names()[0]
else:
    client = get_openai_client()
    model_name = "gpt-5.2"

# Verify OpenAI key
if not use_ai_gateway and not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Please check your .env file.")

# Enable autologging for OpenAI.
# This automatically creates MLflow Traces that capture:
#   - model, temperature, max_tokens (all API params as span attributes)
#   - full input messages and response content (span I/O)
#   - token counts: prompt_tokens, completion_tokens, total_tokens
#   - latency (via span start/end timestamps)
mlflow.openai.autolog()

print("✅ Environment configured successfully")
print(f"   MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"   Using model: {model_name}")
print("   Autolog: ENABLED")

✅ Environment configured successfully
   MLflow Tracking URI: http://localhost:5000
   Using model: gpt-5.2
   Autolog: ENABLED


In [5]:
# Create an experiment
experiment_name = "02-basic-llm-calls"
mlflow.set_experiment(experiment_name)

print(f"📊 Experiment: {experiment_name}")
print("   View in UI: http://localhost:5000")

2026/03/31 14:14:05 INFO mlflow.tracking.fluent: Experiment with name '02-basic-llm-calls' does not exist. Creating a new experiment.


📊 Experiment: 02-basic-llm-calls
   View in UI: http://localhost:5000


In [6]:
# Make a tracked LLM call — autolog does the heavy lifting.
#
# What we do NOT need to log manually (autolog captures all of this):
#   - mlflow.log_param("model", ...)        -> span attribute
#   - mlflow.log_param("temperature", ...)   -> span attribute
#   - mlflow.log_param("max_tokens", ...)    -> span attribute
#   - mlflow.log_metric("latency_seconds")   -> span timestamps
#   - mlflow.log_metric("prompt_tokens")     -> mlflow.chat.tokenUsage
#   - mlflow.log_metric("completion_tokens") -> mlflow.chat.tokenUsage
#   - mlflow.log_metric("total_tokens")      -> mlflow.chat.tokenUsage
#   - mlflow.log_text(prompt, "prompt.txt")  -> span input
#   - mlflow.log_text(answer, "response.txt")-> span output

prompt = "Explain MLflow GenAI Platform in 3-4 sentences."

with mlflow.start_run(run_name="first-llm-tracked-call") as run:

    # The only explicit call: a semantic tag that autolog cannot infer.
    mlflow.set_tag("task", "explanation")

    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0,
        max_completion_tokens=1000
    )

    answer = response.choices[0].message.content

print(f"\n📝 Prompt: {prompt}")
print(f"\n🤖 Response: {answer}")
print(f"\n🔗 Run ID: {run.info.run_id}")
print(f"   View in UI: http://localhost:5000/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}")

🏃 View run first-llm-tracked-call at: http://localhost:5000/#/experiments/2/runs/6e5229b1220a48308a53fe892b34175d
🧪 View experiment at: http://localhost:5000/#/experiments/2

📝 Prompt: Explain MLflow GenAI Platform in 3-4 sentences.

🤖 Response: MLflow GenAI Platform is an extension of MLflow designed to help teams build, evaluate, and deploy generative AI applications—especially LLMs—using a consistent, experiment-driven workflow. It provides tools for tracking prompts, models, parameters, and artifacts; running standardized evaluations (including offline test sets and LLM-as-judge style metrics); and comparing iterations to improve quality and cost. It also supports packaging and serving GenAI components (e.g., prompts, chains, retrievers) so they can be promoted through environments with governance and reproducibility. In practice, it brings MLOps-style rigor—versioning, reproducibility, and monitoring—to the full GenAI lifecycle.

🔗 Run ID: 6e5229b1220a48308a53fe892b34175d
   View 

Trace(trace_id=tr-c58ee8c58ccb1ac37413ffe6cb6eb6b9)

In [7]:
# Simplified helper — autolog captures params, tokens, latency, and I/O automatically.
def simple_llm_call(prompt, model=model_name, temperature=1.0, max_completion_tokens=1000, run_name=None):
    """
    Make an LLM call inside a nested run.
    autolog captures model params, token counts, latency, and full I/O as a Trace.
    The run exists only to group and name the experiment.
    """
    with mlflow.start_run(run_name=run_name, nested=True):
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_completion_tokens=max_completion_tokens
        )
        return response.choices[0].message.content

print("✅ Helper function defined!")

✅ Helper function defined!


In [8]:
mlflow.set_experiment("02-temperature-comparison")

test_prompt = "Write a creative tagline for an AI observability with MLflow GenAI platform."
temperatures = [1.0, 1.5, 2.0]

print("🔬 Running temperature comparison...\n")

# A parent run groups all nested calls together in the UI.
with mlflow.start_run(run_name="temperature-sweep"):
    mlflow.set_tag("sweep_variable", "temperature")

    for temp in temperatures:
        print(f"  temperature={temp} ...")
        response = simple_llm_call(
            prompt=test_prompt,
            model=model_name,
            temperature=temp,
            max_completion_tokens=1000,
            run_name=f"temp_{temp}"
        )
        print(f"    -> {response}\n")

print("✅ Done. Compare traces side-by-side in the MLflow UI.")

2026/03/31 14:34:32 INFO mlflow.tracking.fluent: Experiment with name '02-temperature-comparison' does not exist. Creating a new experiment.


🔬 Running temperature comparison...

  temperature=1.0 ...
🏃 View run temp_1.0 at: http://localhost:5000/#/experiments/3/runs/c42fc1d7f3824644b525ebe40394ac89
🧪 View experiment at: http://localhost:5000/#/experiments/3
    -> **“See every prompt, trace every model, ship GenAI with confidence.”**

  temperature=1.5 ...
🏃 View run temp_1.5 at: http://localhost:5000/#/experiments/3/runs/92efcf85b8c2405f96d5d452df6285f9
🧪 View experiment at: http://localhost:5000/#/experiments/3
    -> **“See every prompt, trace every outcome—MLflow GenAI Observability turns black-box AI into clear, controllable performance.”**

  temperature=2.0 ...
🏃 View run temp_2.0 at: http://localhost:5000/#/experiments/3/runs/7cefb48f863949e2a5e47f49c4deed95
🧪 View experiment at: http://localhost:5000/#/experiments/3
    -> “From prompts to production—full‑stack GenAI observability powered by MLflow.”

🏃 View run temperature-sweep at: http://localhost:5000/#/experiments/3/runs/7ce3e1aef79549a6b4b3d0f90d0b32ce
🧪 View

[Trace(trace_id=tr-038b0c3e0d564dcdc2996b55420e0c19), Trace(trace_id=tr-1f1fbb40a0e975957bf13ba25afddf94), Trace(trace_id=tr-193f2020fcea6f5d23ef6e3b9ebf7298)]

In [ ]:
# Systematic experiment with rich metadata — tags and config artifacts.
mlflow.set_experiment("04-production-candidate-testing")

# Test configurations
open_configs = [
    {
        "name": "baseline",
        "model": "gpt-5-mini",
        "temperature": 1.0,
        "system_prompt": "You are a helpful assistant."
    },
    {
        "name": "creative",
        "model": "gpt-5.2",
        "temperature": 2.0,
        "system_prompt": "You are a creative writing assistant."
    },
]

model_configs =  open_configs
test_prompt = "Explain the concept of LLM temperature."

print("🏷️  Running experiments with semantic tags...\n")

for config in model_configs:
    with mlflow.start_run(run_name=config["name"]):

        # Make the call — autolog captures model, temperature, tokens, I/O, latency.
        response = client.chat.completions.create(
            model=config["model"],
            messages=[
                {"role": "system", "content": config["system_prompt"]},
                {"role": "user", "content": test_prompt}
            ],
            temperature=config["temperature"],
            max_completion_tokens=1000
        )

        # Log only what autolog cannot provide: semantic tags and structured config.
        mlflow.set_tags({
            "config_name": config["name"],
            "task": "explanation",
            "   ": "testing",
            "team": "ai-research",
            "version": "v1.0",
            "production_candidate": str(config["name"] == "baseline").lower(),
        })

        # Save full config as a structured artifact
        mlflow.log_dict(config, "config.json")

        print(f"  ✓ {config['name']} done")

print("\n✅ All runs completed! Filter by tag 'production_candidate=true' in the UI.")

2026/03/31 14:51:51 INFO mlflow.tracking.fluent: Experiment with name '04-production-candidate-testing' does not exist. Creating a new experiment.


🏷️  Running experiments with semantic tags...

  ✓ baseline done
🏃 View run baseline at: http://localhost:5000/#/experiments/4/runs/f80b2bb75dde41ffb8c7b6e8885a12f7
🧪 View experiment at: http://localhost:5000/#/experiments/4
  ✓ creative done
🏃 View run creative at: http://localhost:5000/#/experiments/4/runs/b0f3f83454e84cd2b46413fabe8f0736
🧪 View experiment at: http://localhost:5000/#/experiments/4

✅ All runs completed! Filter by tag 'production_candidate=true' in the UI.


[Trace(trace_id=tr-07f39acd4ae02a264ac728a91a9ebffd), Trace(trace_id=tr-f49f838ee7730ace9aeede561c98d29f)]

🏷️ Tagging Best Practices
Use tags for:

Environment: stage: development/testing/production
Ownership: team: ai-research, owner: jules
Purpose: task: summarization, use_case: customer-support
Status: production_candidate: true, approved: false
Version: version: v1.0, prompt_version: v2.1
Do not duplicate autolog data. Tags like model_used: gpt-5.2 or total_tokens: 342 are already in the Trace. Reserve tags for information that is not derivable from the API call itself.

In [10]:
from mlflow.tracking import MlflowClient

# Use the MlflowClient to query experiments and runs
mlflow_client = MlflowClient()

# Get experiment by name
experiment = mlflow_client.get_experiment_by_name("04-production-candidate-testing")

if experiment:
    print(f"📊 Experiment: {experiment.name}")
    print(f"   ID: {experiment.experiment_id}")

    # Search runs — sort by start_time since autolog stores latency on Traces, not run metrics.
    runs = mlflow_client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["start_time DESC"],
        max_results=5
    )

    print(f"\n   Found {len(runs)} runs:\n" + "="*60)

    for run in runs:
        print(f"\n   Run: {run.info.run_name}")
        if run.data.params:
            print("   Parameters:")
            for key, value in run.data.params.items():
                print(f"      {key}: {value}")
        if run.data.metrics:
            print("   Metrics:")
            for key, value in run.data.metrics.items():
                print(f"      {key}: {value}")
        if run.data.tags.get("config_name"):
            print(f"   Tag config_name: {run.data.tags['config_name']}")
else:
    print("Experiment not found. Make sure you ran the production candidate testing section.")

📊 Experiment: 04-production-candidate-testing
   ID: 4

   Found 2 runs:

   Run: creative
   Tag config_name: creative

   Run: baseline
   Tag config_name: baseline


In [11]:
# Find production candidates using tag filters
if experiment:
    prod_runs = mlflow_client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string="tags.production_candidate = 'true'",
        max_results=5
    )

    print("🏆 Production Candidates:")
    for run in prod_runs:
        print(f"   Name: {run.info.run_name}")
        print(f"   Config: {run.data.tags.get('config_name', 'N/A')}")
        print(f"   Run ID: {run.info.run_id}")

🏆 Production Candidates:
   Name: baseline
   Config: baseline
   Run ID: f80b2bb75dde41ffb8c7b6e8885a12f7


[]